In [1]:
import pandas as pd


In [4]:
# extract only part of data
paysim = pd.read_csv("./data/PS_20174392719_1491204439457_log.csv", nrows=50000)
paysim.to_csv("./data/transactions_sample.csv", index=False)
paysim.shape


(50000, 11)

In [7]:
paysim.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [1]:
import duckdb
con = duckdb.connect("./reconciliation.duckdb")
con.sql("SHOW TABLES").show()

┌───────────────────────────────┐
│             name              │
│            varchar            │
├───────────────────────────────┤
│ deliveries                    │
│ mart_amount_discrepancies     │
│ mart_balance_discrepancies    │
│ mart_duplicate_deliveries     │
│ mart_missing_deliveries       │
│ orders                        │
│ prep_orders_deliveries_merged │
│ stg_deliveries                │
│ stg_orders                    │
│ stg_transactions              │
│ transactions                  │
└───────────────────────────────┘
             11 rows           



In [2]:
con.sql("SELECT * FROM mart_balance_discrepancies").show()


┌─────────────┬─────────────┬──────────┬───────────┬───────────────────────┬──────────────────────┬────────────────────────┬────────────────────────┬──────────┐
│  sender_id  │ receiver_id │   type   │  amount   │ sender_balance_before │ sender_balance_after │ expected_balance_after │      balance_diff      │ is_fraud │
│   varchar   │   varchar   │ varchar  │  double   │        double         │        double        │         double         │         double         │  int64   │
├─────────────┼─────────────┼──────────┼───────────┼───────────────────────┼──────────────────────┼────────────────────────┼────────────────────────┼──────────┤
│ C1265012928 │ M1176932104 │ PAYMENT  │   4024.36 │                2671.0 │                  0.0 │    -1353.3600000000001 │     1353.3600000000001 │        0 │
│ C712410124  │ C195600860  │ DEBIT    │   5337.77 │               41720.0 │             36382.23 │     36382.229999999996 │  7.275957614183426e-12 │        0 │
│ C1900366749 │ C997608398  │ DEBI

In [10]:
con.sql("SELECT count(*) as discrepancies_number FROM mart_balance_discrepancies")

┌──────────────────────┐
│ discrepancies_number │
│        int64         │
├──────────────────────┤
│                39105 │
└──────────────────────┘

In [11]:
con.sql("SELECT count(*) as fraud_discrepancies_number FROM mart_balance_discrepancies WHERE is_fraud > 0")

┌────────────────────────────┐
│ fraud_discrepancies_number │
│           int64            │
├────────────────────────────┤
│                          5 │
└────────────────────────────┘